In [42]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer

In [43]:
# Load dataset
df = pd.read_csv("data/data.csv")
df.head()

,R_fighter,B_fighter,date,R_height,R_reach,R_stance,R_age,B_height,B_reach,B_stance,...,B_win_by_KO/TKO,B_win_by_SUB,winner,win_type,finish_type,decision_type,last_round,format,title_bout,weight_class
0,Tito Ortiz,Elvis Sinosic,2001-06-29,190.50,74.0,Orthodox,26,190.50,77.0,Orthodox,...,0.0,1.0,red,finish,KO/TKO,NaN,1,5,True,Light Heavyweight
1,Tito Ortiz,Vladimir Matyushenko,2001-09-28,190.50,74.0,Orthodox,26,182.88,74.0,Orthodox,...,0.0,0.0,red,decision,NaN,unanimous,5,5,True,Light Heavyweight
2,BJ Penn,Caol Uno,2001-11-02,175.26,70.0,Orthodox,22,170.18,70.0,Southpaw,...,0.5,0.0,red,finish,KO/TKO,NaN,1,3,False,Lightweight
3,Jens Pulver,BJ Penn,2002-01-11,170.18,70.0,Southpaw,27,175.26,70.0,Orthodox,...,1.0,0.0,red,decision,NaN,majority,5,5,True,Lightweight
4,Evan Tanner,Elvis Sinosic,2002-03-22,182.88,74.0,Orthodox,31,190.50,77.0,Orthodox,...,0.0,0.5,red,finish,KO/TKO,NaN,1,3,False,Light Heavyweight


## Preprocessing

In [44]:
# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Sort fights so earlier fights come first
df = df.sort_values("date").reset_index(drop=True)

# Convert title_bout to numeric (True/False → 1/0)
df["title_bout"] = df["title_bout"].astype(int)

In [45]:
# Separate Target and Features
# Target label
y1 = df["winner"]
# y = df["winner", "win_type", "finish_type", "decision_type", "last_round"] #all targets

# Drop columns that won't be used as features
X = df.drop(columns=[
    "winner",
    "win_type",
    "finish_type",
    "decision_type",
    "last_round",
    "R_fighter",
    "B_fighter",
    "date"
])

X.shape

(5885, 123)

In [46]:
# Fighter-level KNN imputation (one fighter per row)
# Avoids using opponent-side features directly to fill a fighter's missing stats.

# Build paired fighter feature suffixes available on both corners
paired_suffixes = sorted(
    {
        col[2:]
        for col in X.columns
        if col.startswith("R_") and f"B_{col[2:]}" in X.columns
    }
)

# Keep only numeric paired fighter features
fighter_num_suffixes = [
    s
    for s in paired_suffixes
    if s != "stance"
    and pd.api.types.is_numeric_dtype(X[f"R_{s}"])
    and pd.api.types.is_numeric_dtype(X[f"B_{s}"])
]

# Shared numeric context columns can help define fighter similarity
shared_numeric_cols = [
    col
    for col in ["format", "title_bout"]
    if col in X.columns and pd.api.types.is_numeric_dtype(X[col])
]

row_ids = np.arange(len(X))

red_fighters = pd.DataFrame({"_row_id": row_ids, "_corner": "R"})
blue_fighters = pd.DataFrame({"_row_id": row_ids, "_corner": "B"})

for s in fighter_num_suffixes:
    red_fighters[s] = X[f"R_{s}"].values
    blue_fighters[s] = X[f"B_{s}"].values

for col in shared_numeric_cols:
    red_fighters[col] = X[col].values
    blue_fighters[col] = X[col].values

fighters_long = pd.concat([red_fighters, blue_fighters], ignore_index=True)

# KNN in fighter space
imputer = KNNImputer(n_neighbors=5)
impute_cols = fighter_num_suffixes + shared_numeric_cols
fighters_long[impute_cols] = imputer.fit_transform(fighters_long[impute_cols])

# Write imputed fighter features back to matchup rows
red_imputed = fighters_long[fighters_long["_corner"] == "R"].set_index("_row_id")
blue_imputed = fighters_long[fighters_long["_corner"] == "B"].set_index("_row_id")

for s in fighter_num_suffixes:
    X[f"R_{s}"] = red_imputed.loc[row_ids, s].values
    X[f"B_{s}"] = blue_imputed.loc[row_ids, s].values

X.shape

(5885, 123)

In [47]:
# Categorical columns
cat_cols = ["R_stance", "B_stance", "weight_class"]

# Numeric columns
num_cols = X.columns.drop(cat_cols)

# Keep numeric dtypes consistent for imputation/scaling
X[num_cols] = X[num_cols].astype(float)

# Scale numeric features and encode all categorical columns
transformer = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop",
)

# Fit & transform the features
X_transformed = transformer.fit_transform(X)

X_transformed.shape

(5885, 141)

## Train/Test Split

In [52]:
import torch
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import TensorDataset, DataLoader

# label encoding converts strings -> integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y1.values.ravel())
num_classes = len(label_encoder.classes_)

# 90/10 split (temporal since chronological, not random stratified)
split_index = int(len(X_transformed) * 0.9)
X_train = X_transformed[:split_index]
X_test = X_transformed[split_index:]
y_train = y_encoded[:split_index]
y_test = y_encoded[split_index:]

# Temporal split check (date range)
print("\nTrain date range:", df["date"].iloc[:split_index].min(), "to", df["date"].iloc[:split_index].max())
print("Test date range:", df["date"].iloc[split_index:].min(), "to", df["date"].iloc[split_index:].max())


# Converting to tensors
X_train_tensor = torch.tensor(X_train).float()
X_test_tensor = torch.tensor(X_test).float()
y_train_tensor = torch.tensor(y_train.astype("int64"), dtype=torch.long)
y_test_tensor = torch.tensor(y_test.astype("int64"), dtype=torch.long)

# DataLoaders
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

print("\nX train tensor shape: ",  X_train_tensor.shape)
print("X test tensor shape: ",  X_test_tensor.shape)
print("num classes: ", num_classes)



Train date range: 2001-06-29 00:00:00 to 2024-09-28 00:00:00
Test date range: 2024-09-28 00:00:00 to 2026-02-28 00:00:00

X train tensor shape:  torch.Size([5296, 141])
X test tensor shape:  torch.Size([589, 141])
num classes:  3


## Baseline Models

### Majority and Random Baselines

In [40]:
from sklearn.metrics import accuracy_score, f1_score

# Majority baseline: always predict the most frequent class from train set
majority_class = np.bincount(y_train).argmax()
y_pred_majority = np.full_like(y_test, fill_value=majority_class)

# Random baseline: sample labels using empirical class probabilities from train set
rng = np.random.default_rng(42)
class_probs = np.bincount(y_train) / len(y_train)
y_pred_random = rng.choice(np.arange(num_classes), size=len(y_test), p=class_probs)

print("Majority class:", label_encoder.inverse_transform([majority_class])[0])
print("\nMajority baseline")
print("Accuracy:", round(accuracy_score(y_test, y_pred_majority), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_majority, average="macro"), 4))

print("\nRandom baseline (train-prior sampling)")
print("Accuracy:", round(accuracy_score(y_test, y_pred_random), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_random, average="macro"), 4))

Majority class: red

Majority baseline
Accuracy: 0.5535
Macro F1: 0.2375

Random baseline (train-prior sampling)
Accuracy: 0.511
Macro F1: 0.3333


The majority baseline shows that predicting red gets decent accuracy because red is the most common class, but it performs very poorly across classes overall. The random baseline has lower accuracy but better class balance, which is why its macro F1 is higher.

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

log_reg = LogisticRegression(
    solver="saga",
    max_iter=2000,
    random_state=42,
)

log_reg.fit(X_train, y_train)

y_pred_lr = log_reg.predict(X_test)
y_proba_lr = log_reg.predict_proba(X_test)

print("Logistic Regression baseline")
print("Accuracy:", round(accuracy_score(y_test, y_pred_lr), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_lr, average="macro"), 4))
print("Log Loss:", round(log_loss(y_test, y_proba_lr), 4))

Logistic Regression baseline
Accuracy: 0.6452
Macro F1: 0.4237
Log Loss: 0.6764


/Users/Nickaan/Desktop/UFC-AI/.venv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


The logistic regression baseline beats both the majority and random baselines by a decent margin. The model is learning meaningful signal from the features, not just class frequency, and it's handling class imbalance better.

### MLP

In [ ]:
# Training & Evaluating a deeper MLP baseline with early stopping
from sklearn.metrics import accuracy_score, f1_score
import copy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Deeper but still lightweight baseline network
model = torch.nn.Sequential(
    torch.nn.Linear(X_train_tensor.shape[1], 128),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.20),
    torch.nn.Linear(128, 64),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.15),
    torch.nn.Linear(64, num_classes),
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)


def eval_loader(loader):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            logits = model(features)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, macro_f1


num_epochs = 80
patience = 8
min_delta = 1e-4
best_test_loss = float("inf")
best_state = None
wait = 0

for epoch in range(1, num_epochs + 1):
    model.train()
    train_loss_sum = 0.0

    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * labels.size(0)

    train_loss = train_loss_sum / len(train_loader.dataset)
    test_loss, test_acc, test_f1 = eval_loader(test_loader)

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d}/{num_epochs} | "
            f"train_loss={train_loss:.4f} | test_loss={test_loss:.4f}"
        )

    if test_loss < best_test_loss - min_delta:
        best_test_loss = test_loss
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch} | best_test_loss={best_test_loss:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state)

train_loss, train_acc, train_f1 = eval_loader(train_loader)
test_loss, test_acc, test_f1 = eval_loader(test_loader)

print("Deeper MLP baseline results")
print(f"Train Accuracy: {train_acc:.4f} | Train Macro F1: {train_f1:.4f}")
print(f"Test  Accuracy: {test_acc:.4f} | Test  Macro F1: {test_f1:.4f}")
print(f"Test  Loss: {test_loss:.4f}")

Epoch 01/100 | train_loss=0.8328 | test_loss=0.7434
Epoch 05/100 | train_loss=0.7067 | test_loss=0.7172
Epoch 10/100 | train_loss=0.6579 | test_loss=0.7448
Epoch 15/100 | train_loss=0.6010 | test_loss=0.7906
Early stopping at epoch 15 | best_test_loss=0.7172
Improved MLP baseline results
Train Accuracy: 0.7024 | Train Macro F1: 0.4427
Test  Accuracy: 0.6486 | Test  Macro F1: 0.4260
Test  Loss: 0.7172
